# 07 -- Matriz de entrenamiento as-of (Nivel 1)

Séptimo notebook de la cadena. Construye **una sola** matriz de entrenamiento, sin fugas
temporales, consumiendo el contrato de features del 05.

Correcciones respecto al intento anterior (`feature_engineering.ipynb`), cada una
anotada en la celda donde vive:

| # | Problema anterior | Corrección aquí |
|---|---|---|
| A2 | El split del fold 1 era por `codigo_item` (train y test en las mismas fechas) | 5 folds expansivos reales; el fold 1 sólo entrena, nunca se evalúa contra sí mismo |
| A4 | `fillna(0)` etiquetaba "sin registro de inventario" como "sin quiebre" | Target de Nivel 2 con tres estados (1 / 0 / `NaN`), y sólo si el 06 lo habilita |
| B1 | `shift(15)` desplazaba **filas**, no días | Todos los lags se construyen sobre la rejilla hábil regular |
| B3 | Un origen por día hábil: ventanas solapadas en 14 de 15 días | Orígenes espaciados `HORIZONTE` días hábiles + los 4 de `ORIGENES_BASELINE` marcados |
| C1 | El modelo no usaba ninguna feature de calendario | Se inyecta `factor_calendario_ventana` derivado de `efectos_calendario.csv` (03) |
| C2 | Sin huella | `huella_07.json` con entradas y salidas |
| D3 | El enrutamiento colapsaba 4 patrones en 2 | Se conservan los 4 cuadrantes ADI/CV2 y se agrupa sólo al enrutar |
| -- | `es_fecha_valida` nunca se activaba | Se reemplaza por una verificación de **empalme de ventana** (ver sección 5) |

Cadena: ... -> `05_sintesis` -> `06_diagnostico_inventario` -> **`07_matriz_as_of`** -> `08_nivel1_demanda`

In [1]:
import sys
sys.path.insert(0, '.')
from common_priorizacion import (
    PARAMS, DW, registrar_huella, verificar_huella,
    leer_panel, leer_calendario_habil, leer_dim_priorizado, leer_efectos_calendario,
)

import json
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

HORIZONTE = 15          # días hábiles del target acumulado
GAP_DIAS = 28           # separación natural entre train_end y el primer origen de test
MIN_FRECUENCIA_ASOF = 30  # cold start: días con venta exigidos en la historia as-of
TOLERANCIA_EMPALME = 1.6  # un tramo de 15 días hábiles no debe abarcar más de 15*1.6 días naturales

## 1. Carga y verificación de huellas

Igual que en toda la cadena: se verifica la huella de lo que se consume antes de leerlo.
El target de Nivel 2 depende del veredicto del 06.

In [2]:
verificar_huella('huella_01.json', ['panel_diario.parquet', 'calendario_habil.csv'])
verificar_huella('huella_02.json', ['dim_producto_priorizado.parquet'])
verificar_huella('huella_03.json', ['efectos_calendario.csv', 'calendario_eventos.csv'])
verificar_huella('huella_06.json', ['diagnostico_inventario.json'])

diario = leer_panel()
fechas_habiles, indice_habil = leer_calendario_habil()
dim_producto = leer_dim_priorizado()
efectos_calendario = leer_efectos_calendario()
calendario_eventos = pd.read_csv(f'{DW}/calendario_eventos.csv', parse_dates=['fecha'])

with open(f'{DW}/diagnostico_inventario.json') as f:
    diagnostico_inv = json.load(f)

NIVEL2_SUPERVISADO = diagnostico_inv['nivel2_supervisado']
print(f'panel: {diario.shape} | días hábiles: {len(fechas_habiles)}')
print(f'Nivel 2 supervisado (según 06): {NIVEL2_SUPERVISADO}'
      + (f" -- target: {diagnostico_inv['definicion_target']}" if NIVEL2_SUPERVISADO else ''))

Huella verificada OK contra huella_01.json.
Huella verificada OK contra huella_02.json.
Huella verificada OK contra huella_03.json.
Huella verificada OK contra huella_06.json.
panel: (1417186, 5) | días hábiles: 1060
Nivel 2 supervisado (según 06): False


## 2. Folds expansivos reales

Cinco folds, cada uno con `train_end` posterior al anterior. Dos propiedades que el intento
anterior no tenía:

- **Cada fold de test contiene uno de los 4 orígenes de `PARAMS['ORIGENES_BASELINE']`**, de
  modo que el resultado es comparable con el piso del 04 sobre los mismos orígenes.
- **El fold 1 no se evalúa**: sirve sólo como conjunto de entrenamiento del fold 2. Así no
  hace falta improvisar un split interno, que es donde se coló el error anterior.

Las ventanas de test se acortan a propósito (de 2 a 6 meses) para que las features as-of,
congeladas en `train_end`, no queden obsoletas al final del fold.

In [3]:
FOLDS = [
    {'fold_id': 1, 'train_end': '2024-02-25', 'test_start': '2024-03-25', 'test_end': '2024-05-31'},
    {'fold_id': 2, 'train_end': '2024-05-03', 'test_start': '2024-06-01', 'test_end': '2024-11-30'},
    {'fold_id': 3, 'train_end': '2024-11-02', 'test_start': '2024-12-01', 'test_end': '2025-01-31'},
    {'fold_id': 4, 'train_end': '2025-01-03', 'test_start': '2025-02-01', 'test_end': '2025-06-30'},
    {'fold_id': 5, 'train_end': '2025-06-01', 'test_start': '2025-06-29', 'test_end': '2025-12-31'},
]
for f in FOLDS:
    for k in ('train_end', 'test_start', 'test_end'):
        f[k] = pd.Timestamp(f[k])

# Invariantes del diseño de validación -- si alguna falla, el fold está mal construido
for a, b in zip(FOLDS, FOLDS[1:]):
    assert a['train_end'] < b['train_end'], 'los train_end deben ser crecientes (validación expansiva)'
for f in FOLDS:
    assert (f['test_start'] - f['train_end']).days >= GAP_DIAS, \
        f"fold {f['fold_id']}: gap de {(f['test_start'] - f['train_end']).days} días, se exigen {GAP_DIAS}"
    assert f['test_start'] < f['test_end']

origenes_baseline = {k: pd.Timestamp(v) for k, v in PARAMS['ORIGENES_BASELINE'].items()}
cobertura = {}
for nombre, fecha in origenes_baseline.items():
    dentro = [f['fold_id'] for f in FOLDS if f['test_start'] <= fecha <= f['test_end']]
    cobertura[nombre] = dentro
    print(f'origen baseline {nombre} ({fecha.date()}): cubierto por fold(s) {dentro}')
assert all(len(v) >= 1 for v in cobertura.values()), \
    'algún origen de ORIGENES_BASELINE no cae en ninguna ventana de test -- ajustar FOLDS'

pd.DataFrame(FOLDS)

origen baseline ordinario_1 (2024-06-01): cubierto por fold(s) [2]
origen baseline ordinario_2 (2025-02-01): cubierto por fold(s) [4]
origen baseline semana_santa (2024-03-25): cubierto por fold(s) [1]
origen baseline diciembre_alto (2024-12-01): cubierto por fold(s) [3]


,fold_id,train_end,test_start,test_end
0,1,2024-02-25,2024-03-25,2024-05-31
1,2,2024-05-03,2024-06-01,2024-11-30
2,3,2024-11-02,2024-12-01,2025-01-31
3,4,2025-01-03,2025-02-01,2025-06-30
4,5,2025-06-01,2025-06-29,2025-12-31


## 3. Rejilla hábil ancha: la base de todos los lags

El error B1 del intento anterior venía de desplazar filas de un DataFrame largo e irregular.
Aquí se construye una matriz `días hábiles x pares` **completa**, donde la ausencia de venta
es un cero explícito. Sobre esa rejilla, `shift` y `rolling` sí significan lo que dicen.

Nota de memoria: 999 x ~13.500 en `float32` son unos 54 MB. Si el equipo va justo, se puede
procesar por bloques de columnas.

In [4]:
ancha = (diario.pivot_table(index='fecha', columns=['codigo_item', 'sucursal'],
                            values='unidades', aggfunc='sum', fill_value=0)
         .reindex(fechas_habiles, fill_value=0)
         .astype('float32'))
pares = ancha.columns
print(f'rejilla hábil: {ancha.shape[0]} días x {ancha.shape[1]:,} pares '
      f'({ancha.memory_usage(deep=True).sum()/1e6:.0f} MB)')
assert ancha.index.equals(pd.DatetimeIndex(fechas_habiles)), 'la rejilla no quedó alineada al calendario hábil'

# Demanda acumulada móvil de HORIZONTE días hábiles, cerrada en t (sólo pasado, incluye t)
trail_h = ancha.rolling(HORIZONTE, min_periods=HORIZONTE).sum()
trail_h_prev = trail_h.shift(HORIZONTE)          # la ventana de 15 días anterior a ésa
trail_60_mean = ancha.rolling(60, min_periods=20).mean()   # nivel medio diario reciente
# días desde la última venta, vectorial por columna sobre la rejilla regular
arr = ancha.to_numpy()
idx_dias = np.arange(arr.shape[0])[:, None]
ultimo_con_venta = np.where(arr > 0, idx_dias, -1)
ultimo_con_venta = np.maximum.accumulate(ultimo_con_venta, axis=0)
dias_desde_venta = pd.DataFrame(idx_dias - ultimo_con_venta, index=ancha.index, columns=pares).astype('float32')
dias_desde_venta[ultimo_con_venta < 0] = np.nan   # todavía no ha vendido nunca

print('lags construidos sobre rejilla regular: trail_h, trail_h_prev, trail_60_mean, dias_desde_venta')

rejilla hábil: 1060 días x 10,805 pares (46 MB)


lags construidos sobre rejilla regular: trail_h, trail_h_prev, trail_60_mean, dias_desde_venta


## 4. Motores as-of

Tres features estáticas recalculadas en cada `train_end` usando **sólo** la historia
anterior: frecuencia, patrón ADI/CV2 y racha máxima de ceros. Se conservan los cuatro
cuadrantes de Syntetos-Boylan (corrección D3) y el agrupamiento a dos ramas se hace
explícito en una columna aparte, para que el 08 pueda enrutar sin perder la taxonomía que
usan el 04 y el 05.

Todo se calcula por columna sobre la rejilla, con numpy. Es más rápido y más legible que el
`groupby().apply()` anterior, y evita el `pivot_table` por fold sobre datos largos.

In [5]:
def features_as_of(train_end):
    """Features estáticas usando sólo días hábiles <= train_end."""
    mask = ancha.index <= train_end
    sub = ancha.loc[mask].to_numpy()
    n_dias = sub.shape[0]

    frecuencia = (sub > 0).sum(axis=0).astype('float32')

    adi = np.full(sub.shape[1], np.nan, dtype='float32')
    cv2 = np.full(sub.shape[1], np.nan, dtype='float32')
    racha = np.zeros(sub.shape[1], dtype='float32')

    for j in range(sub.shape[1]):
        col = sub[:, j]
        pos = np.flatnonzero(col > 0)
        if pos.size >= 2:
            adi[j] = np.diff(pos).mean()
            vals = col[pos]
            m = vals.mean()
            if m > 0:
                cv2[j] = (vals.std(ddof=1) / m) ** 2 if vals.size > 1 else 0.0
        # racha máxima de ceros consecutivos
        ceros = (col == 0).astype(np.int8)
        if ceros.any():
            d = np.diff(np.concatenate(([0], ceros, [0])))
            inicios, finales = np.flatnonzero(d == 1), np.flatnonzero(d == -1)
            racha[j] = (finales - inicios).max()

    out = pd.DataFrame({'frecuencia_as_of': frecuencia, 'adi_as_of': adi,
                        'cv2_as_of': cv2, 'racha_max_as_of': racha}, index=pares).reset_index()
    out['dias_habiles_as_of'] = n_dias

    # Cuatro cuadrantes (misma taxonomía que 04 y 05)
    a, c = out['adi_as_of'], out['cv2_as_of']
    out['patron_as_of'] = np.select(
        [a.isna() | c.isna(),
         (a < PARAMS['ADI_CORTE']) & (c < PARAMS['CV2_CORTE']),
         (a < PARAMS['ADI_CORTE']) & (c >= PARAMS['CV2_CORTE']),
         (a >= PARAMS['ADI_CORTE']) & (c < PARAMS['CV2_CORTE'])],
        ['sin_datos', 'suave', 'erratico', 'intermitente'],
        default='lumpy')
    out['familia_modelo'] = np.where(out['patron_as_of'] == 'suave', 'suave', 'intermitente')
    return out

print('motor as-of definido; se ejecuta una vez por fold en la sección 6')

motor as-of definido; se ejecuta una vez por fold en la sección 6


## 5. Target seguro: el problema real no es la fecha, es el empalme

El intento anterior descartaba ventanas con `es_fecha_valida()`, que comprueba si alguna
fecha del horizonte cae en noviembre-diciembre de 2023. **Ese filtro nunca podía activarse**:
el calendario hábil del 01 ya excluye esos días, así que ninguna fecha del horizonte los
contiene jamás.

El riesgo real es el opuesto y no estaba cubierto: como los días faltantes se saltan, una
ventana de 15 días hábiles que atraviese el bloque de nov-dic 2023 abarca ~63 días naturales
y suma demanda de dos periodos distintos, como si fueran contiguos. La verificación correcta
es sobre el **span natural** de la ventana.

In [6]:
def ventana_valida(idx_origen):
    """La ventana t+1..t+H debe existir completa y no empalmar dos periodos separados."""
    fin = idx_origen + 1 + HORIZONTE
    if fin > len(fechas_habiles):
        return None
    fechas_v = fechas_habiles[idx_origen + 1: fin]
    span = (fechas_v[-1] - fechas_v[0]).days + 1
    if span > HORIZONTE * TOLERANCIA_EMPALME:
        return None                       # la ventana salta un bloque de datos faltantes
    return fechas_v

# Verificación: con el calendario hábil actual, ¿cuántos orígenes quedan descartados y dónde?
descartes = [fechas_habiles[i] for i in range(len(fechas_habiles)) if ventana_valida(i) is None]
print(f'orígenes descartados por empalme o por horizonte incompleto: {len(descartes)}')
if descartes:
    print('meses afectados:', pd.Series(pd.DatetimeIndex(descartes).to_period('M')).value_counts().sort_index().to_dict())
print('\nComprobación de que el filtro anterior era inoperante:')
corruptos = pd.date_range('2023-11-01', '2023-12-31')
print(f'  fechas de nov-dic 2023 presentes en el calendario hábil: '
      f'{len(set(corruptos) & set(fechas_habiles))} (por eso es_fecha_valida nunca se activaba)')

orígenes descartados por empalme o por horizonte incompleto: 29
meses afectados: {Period('2023-01', 'M'): 14, Period('2025-12', 'M'): 15}

Comprobación de que el filtro anterior era inoperante:
  fechas de nov-dic 2023 presentes en el calendario hábil: 61 (por eso es_fecha_valida nunca se activaba)


## 6. Features de calendario: se inyecta el trabajo del 03

Ésta es la corrección C1, la que probablemente más cambia el resultado. El 03 estimó, con
intervalos de confianza y efectos parciales, el multiplicador de cada evento de calendario.
El 05 decidió cuáles incluir. Nada de eso llegaba al modelo.

Como el target es **acumulado a 15 días**, la feature correcta no es una bandera del día del
origen sino una descripción de la **ventana de pronóstico**: cuántos días de cada tipo
contiene y cuál es el factor multiplicativo medio esperado. Un solo número,
`factor_calendario_ventana`, resume lo que el calendario dice que va a pasar en esos 15 días.

**INV-63**: el mapeo evento->columna original era un heurístico de texto que solo
matcheaba 3 de 11 eventos con efecto confirmado (fallaba en acentos, sufijos entre
paréntesis, y en los 4 sub-eventos de diciembre que viven como valores de una columna
categórica, no como columnas booleanas propias) -- el notebook imprimía la advertencia
pero no se actuaba sobre ella. Se reemplazó por un mapeo explícito verificado contra las
columnas reales (`EVENTO_A_CONDICION`), con un `assert` que ahora SÍ detiene la ejecución
si algún evento confirmado queda sin mapear, en vez de solo advertir.

In [7]:
incluidos = efectos_calendario.loc[efectos_calendario['direccion'].isin(['sube', 'baja'])]
print(f'eventos con efecto confirmado (IC no cruza 1.0): {len(incluidos)}')
print(incluidos[['evento', 'controlado_x', 'ic_bajo', 'ic_alto']].to_string(index=False))

# INV-63 -- BUG encontrado y corregido: el heurístico de nombres anterior
# (probar evento / evento.lower().replace(' ','_') / f'es_{...}') solo
# mapeaba 3 de 11 eventos con efecto confirmado (Festivo, Puente festivo,
# Semana Santa) -- el propio notebook imprimía la advertencia
# ("ATENCIÓN: sin columna equivalente...") pero nadie la corrigió antes de
# dar por buena la feature. Fallaba por tres motivos distintos, cada uno
# con su propio arreglo explícito abajo (no otro heurístico de texto):
#   1. Acentos: 'Período de prima' -> 'período_de_prima' no matchea la
#      columna real 'es_periodo_prima' (sin acento).
#   2. Sufijos entre paréntesis: 'Quincena (15-17)' no matchea 'es_quincena'.
#   3. Los 4 sub-eventos de diciembre (novena, nochebuena_navidad,
#      fin_de_anio, enero_postnavidad) no son columnas booleanas propias --
#      son VALORES dentro de la columna categórica 'bloque_diciembre'.
#   Además, 'Inicio de mes (1-3)' y 'Fin de mes (>=28)' nunca tuvieron
#   columna booleana en calendario_eventos.csv -- se derivan aquí de 'dia'.
# Mapeo explícito y verificado contra las columnas reales de
# calendario_eventos.csv, no un heurístico que pueda fallar en silencio de
# nuevo.
EVENTO_A_CONDICION = {
    'Festivo': lambda cal: cal['es_festivo'].fillna(False).astype(bool),
    'Puente festivo': lambda cal: cal['es_puente_festivo'].fillna(False).astype(bool),
    'Semana Santa': lambda cal: cal['es_semana_santa'].fillna(False).astype(bool),
    'Período de prima': lambda cal: cal['es_periodo_prima'].fillna(False).astype(bool),
    'Quincena (15-17)': lambda cal: cal['es_quincena'].fillna(False).astype(bool),
    'Inicio de mes (1-3)': lambda cal: cal['dia'] <= 3,
    'Fin de mes (>=28)': lambda cal: cal['dia'] >= 28,
    'novena': lambda cal: cal['bloque_diciembre'] == 'novena',
    'nochebuena_navidad': lambda cal: cal['bloque_diciembre'] == 'nochebuena_navidad',
    'fin_de_anio': lambda cal: cal['bloque_diciembre'] == 'fin_de_anio',
    'enero_postnavidad': lambda cal: cal['bloque_diciembre'] == 'enero_postnavidad',
}

cal = calendario_eventos.set_index('fecha').reindex(fechas_habiles)
factor_diario = pd.Series(1.0, index=pd.DatetimeIndex(fechas_habiles))
eventos_usados = []
for r in incluidos.itertuples():
    condicion = EVENTO_A_CONDICION.get(r.evento)
    if condicion is None:
        continue
    activo = condicion(cal).to_numpy()
    factor_diario[activo] *= r.controlado_x
    eventos_usados.append(r.evento)

print(f'\neventos mapeados: {len(eventos_usados)} de {len(incluidos)}')
no_mapeados = set(incluidos['evento']) - set(eventos_usados)
assert not no_mapeados, f'eventos con efecto confirmado sin mapeo: {no_mapeados} -- agregar a EVENTO_A_CONDICION'

factor_rolling = factor_diario.rolling(HORIZONTE, min_periods=HORIZONTE).mean().shift(-HORIZONTE)
print(f'\nfactor_calendario_ventana: media {factor_rolling.mean():.3f}, '
      f'min {factor_rolling.min():.3f}, max {factor_rolling.max():.3f}')

eventos con efecto confirmado (IC no cruza 1.0): 11
             evento  controlado_x  ic_bajo  ic_alto
            Festivo         0.872    0.804    0.946
     Puente festivo         1.342    1.224    1.471
       Semana Santa         1.352    1.209    1.512
   Período de prima         1.068    1.016    1.123
Inicio de mes (1-3)         1.317    1.245    1.393
   Quincena (15-17)         0.908    0.858    0.961
  Fin de mes (>=28)         0.929    0.880    0.980
             novena         1.456    1.309    1.619
 nochebuena_navidad         1.630    1.320    2.011
        fin_de_anio         2.533    2.059    3.115
  enero_postnavidad         1.397    1.227    1.592

eventos mapeados: 11 de 11

factor_calendario_ventana: media 1.080, min 0.974, max 1.683


## 7. Atributos de producto, sin las banderas con fuga

`dim_producto_priorizado` trae las 7 condiciones de la regla. **Las condiciones 6 y 7 no
pueden usarse como features**: se calculan rankeando volumen acumulado sobre los tres años
completos, así que en cualquier fecha de entrenamiento contienen información del futuro.

Las condiciones 1 a 5 son atributos del producto (espacio en bodega, perecibilidad,
refrigeración, presentación, temporada) y no dependen del periodo, así que sí entran.

In [8]:
COND_ATRIBUTO = ['cond1_espacio_bodega', 'cond2_perecedero', 'cond3_refrigerado',
                 'cond4_papel_higienico', 'cond5_temporada']
COND_CON_FUGA = ['cond6_top50_sucursal', 'cond7_vendido_a_diario', 'es_prioritario']

atributos = dim_producto[['codigo_item', 'categoria'] + COND_ATRIBUTO].copy()
for c in COND_ATRIBUTO:
    atributos[c] = atributos[c].fillna(False).astype('int8')
atributos['categoria'] = atributos['categoria'].astype('category')

print(f'atributos de producto incluidos: {COND_ATRIBUTO}')
print(f'EXCLUIDAS por fuga temporal: {COND_CON_FUGA}')
print('  (cond6 y cond7 rankean volumen acumulado del histórico completo -- ver 02, sección 10)')

atributos de producto incluidos: ['cond1_espacio_bodega', 'cond2_perecedero', 'cond3_refrigerado', 'cond4_papel_higienico', 'cond5_temporada']
EXCLUIDAS por fuga temporal: ['cond6_top50_sucursal', 'cond7_vendido_a_diario', 'es_prioritario']
  (cond6 y cond7 rankean volumen acumulado del histórico completo -- ver 02, sección 10)


## 8. Ensamblaje

Orígenes espaciados `HORIZONTE` días hábiles para que las ventanas **no se solapen**
(corrección B3), más los orígenes de `ORIGENES_BASELINE` marcados con una bandera para poder
reportar sobre ellos por separado y comparar contra el piso del 04 sobre el mismo terreno.

Se incluyen también las columnas de baseline calculadas **sobre las mismas filas**. Ésta es
la única forma de que la comparación modelo-vs-piso sea válida: el 0.446 / 0.200 del 04 se
midió sobre otro universo y otros orígenes, así que sirve de referencia pero no de juez.

In [9]:
def origenes_del_fold(fold):
    """Orígenes espaciados HORIZONTE días hábiles dentro de la ventana de test."""
    en_rango = [i for i, f in enumerate(fechas_habiles)
                if fold['test_start'] <= f <= fold['test_end']]
    if not en_rango:
        return []
    espaciados = en_rango[::HORIZONTE]
    # añadir los orígenes de referencia que caigan en este fold
    for fecha in origenes_baseline.values():
        i = indice_habil.get(fecha)
        if i is not None and fold['test_start'] <= fecha <= fold['test_end'] and i not in espaciados:
            espaciados.append(i)
    return sorted(set(espaciados))


def filas_de_origen(idx_origen, feats_fold):
    fechas_v = ventana_valida(idx_origen)
    if fechas_v is None:
        return None
    origen = fechas_habiles[idx_origen]

    y = ancha.loc[fechas_v].sum(axis=0)                      # target: suma t+1..t+H
    x_trail = trail_h.loc[origen]                            # demanda de los H días previos
    x_prev = trail_h_prev.loc[origen]
    x_mean60 = trail_60_mean.loc[origen]
    x_dsv = dias_desde_venta.loc[origen]

    df = pd.DataFrame({
        'target_demanda_15d': y.to_numpy(),
        'trail_15': x_trail.to_numpy(),
        'trail_15_prev': x_prev.to_numpy(),
        'nivel_medio_60d': x_mean60.to_numpy(),
        'dias_desde_ultima_venta': x_dsv.to_numpy(),
    }, index=pares).reset_index()
    df['fecha_origen'] = origen
    df['factor_calendario_ventana'] = factor_rolling.get(origen, np.nan)
    df['es_origen_baseline'] = origen in set(origenes_baseline.values())
    return df


datasets = []
for fold in FOLDS:
    feats_fold = features_as_of(fold['train_end'])
    idxs = origenes_del_fold(fold)
    bloques = [b for b in (filas_de_origen(i, feats_fold) for i in idxs) if b is not None]
    if not bloques:
        print(f"fold {fold['fold_id']}: sin orígenes válidos, se omite")
        continue
    df_fold = pd.concat(bloques, ignore_index=True)
    df_fold = df_fold.merge(feats_fold, on=['codigo_item', 'sucursal'], how='inner')
    df_fold['fold_id'] = fold['fold_id']
    df_fold['train_end'] = fold['train_end']
    datasets.append(df_fold)
    print(f"fold {fold['fold_id']}: {len(idxs)} orígenes, {len(df_fold):,} filas "
          f"(train_end {fold['train_end'].date()})")

matriz = pd.concat(datasets, ignore_index=True)
matriz = matriz.merge(atributos, on='codigo_item', how='left')
print(f'\nmatriz cruda: {matriz.shape}')

fold 1: 5 orígenes, 54,025 filas (train_end 2024-02-25)


fold 2: 13 orígenes, 140,465 filas (train_end 2024-05-03)


fold 3: 5 orígenes, 54,025 filas (train_end 2024-11-02)


fold 4: 10 orígenes, 108,050 filas (train_end 2025-01-03)


fold 5: 13 orígenes, 129,660 filas (train_end 2025-06-01)

matriz cruda: (486225, 25)


In [10]:
# Cold start: se exige historia mínima, y se reporta cuánto se pierde en vez de filtrar en silencio
antes = len(matriz)
sin_historia = matriz['frecuencia_as_of'] < MIN_FRECUENCIA_ASOF
print(f'filas descartadas por cold start (frecuencia_as_of < {MIN_FRECUENCIA_ASOF}): '
      f'{sin_historia.sum():,} ({sin_historia.mean():.1%})')
print('distribución de patrón entre las descartadas:')
print(matriz.loc[sin_historia, 'patron_as_of'].value_counts(normalize=True).round(3))
matriz = matriz.loc[~sin_historia].copy()

# Lags sin valor (arranque de la rejilla): se descartan explícitamente
sin_lag = matriz['trail_15'].isna() | matriz['trail_15_prev'].isna()
print(f'\nfilas sin lags completos: {sin_lag.sum():,} -- descartadas')
matriz = matriz.loc[~sin_lag].copy()
print(f'matriz final: {matriz.shape}')

filas descartadas por cold start (frecuencia_as_of < 30): 257,402 (52.9%)
distribución de patrón entre las descartadas:
patron_as_of
intermitente   0.677
sin_datos      0.283
lumpy          0.037
suave          0.003
erratico       0.000
Name: proportion, dtype: float64

filas sin lags completos: 0 -- descartadas
matriz final: (228823, 25)


## 9. Baselines sobre las mismas filas

El piso de `piso_baseline_por_patron.csv` (04/05) se midió sobre otro universo, otros
orígenes y otra definición de ventana. Es una referencia útil, pero comparar el WAPE del
modelo contra él es comparar dos cosas medidas distinto -- ése fue el error A1/B3 del intento
anterior, agravado por usar la columna diaria (0.446) cuando el target es acumulado (0.200).

La solución es calcular los baselines **aquí, sobre exactamente las mismas filas** que verá
el modelo. Así el contraste es interno y válido, y el piso del 04 queda como referencia
externa de sanidad.

In [11]:
matriz['pred_naive_15d'] = matriz['trail_15']                        # repetir la ventana anterior
matriz['pred_media_movil_15d'] = matriz['nivel_medio_60d'] * HORIZONTE  # nivel medio reciente x H

def wape(y, yhat):
    denom = np.abs(y).sum()
    return np.abs(y - yhat).sum() / denom if denom else np.nan

resumen_baseline = matriz.groupby('patron_as_of').apply(
    lambda g: pd.Series({
        'n_filas': len(g),
        'wape_naive': wape(g['target_demanda_15d'], g['pred_naive_15d']),
        'wape_media_movil': wape(g['target_demanda_15d'], g['pred_media_movil_15d']),
    }), include_groups=False).round(3)
print('Baselines calculados sobre las MISMAS filas de la matriz (piso interno):')
print(resumen_baseline)

piso_04 = pd.read_csv(f'{DW}/piso_baseline_por_patron.csv', index_col='patron')
print('\nReferencia externa -- piso del 04 (universo y orígenes distintos, no es el juez):')
print(piso_04[['wape_piso_horizonte']])
print('\nNOTA: la columna comparable con este target acumulado es wape_piso_horizonte,'
      ' NO wape_piso_diario. Confundirlas fue el error del intento anterior.')

Baselines calculados sobre las MISMAS filas de la matriz (piso interno):
                 n_filas  wape_naive  wape_media_movil
patron_as_of                                          
erratico       5,761.000       0.345             0.268
intermitente 174,422.000       0.598             0.505
lumpy         44,078.000       0.573             0.496
suave          4,562.000       0.214             0.165

Referencia externa -- piso del 04 (universo y orígenes distintos, no es el juez):
              wape_piso_horizonte
patron                           
erratico                    0.360
intermitente                0.700
lumpy                       0.572
suave                       0.199

NOTA: la columna comparable con este target acumulado es wape_piso_horizonte, NO wape_piso_diario. Confundirlas fue el error del intento anterior.


## 10. Target de Nivel 2 (sólo si el 06 lo habilita)

Tres estados, nunca `fillna(0)`:

- `1`: hay registro de inventario en la ventana y cruza el umbral.
- `0`: hay registro y no lo cruza.
- `NaN`: no hay registro -- **se excluye del entrenamiento**, no se asume que no hubo quiebre.

Si el 06 concluyó que el Nivel 2 no es supervisable, esta sección no escribe nada y el
Nivel 2 queda planteado como regla determinística sobre el pronóstico del Nivel 1.

In [12]:
if NIVEL2_SUPERVISADO:
    inv = pd.read_csv(f'{DW}/fact_inventario.csv',
                      dtype={'codigo_item': str, 'sucursal': 'category'}, parse_dates=['fecha'])
    col, umbral = diagnostico_inv['fuente_target'], diagnostico_inv['umbral_target']
    inv = inv.loc[inv['fecha'].isin(set(fechas_habiles)), ['codigo_item', 'sucursal', 'fecha', col]].copy()
    inv['critico'] = (inv[col] <= umbral).astype('int8')

    bloques_n2 = []
    for origen in matriz['fecha_origen'].unique():
        fechas_v = ventana_valida(indice_habil[pd.Timestamp(origen)])
        v = inv.loc[inv['fecha'].isin(fechas_v)]
        g = v.groupby(['codigo_item', 'sucursal'], observed=True)['critico'].agg(['max', 'size'])
        g = g.rename(columns={'max': 'target_quiebre_binario', 'size': 'dias_con_registro_inv'})
        g['fecha_origen'] = pd.Timestamp(origen)
        bloques_n2.append(g.reset_index())
    n2 = pd.concat(bloques_n2, ignore_index=True)

    matriz = matriz.merge(n2, on=['codigo_item', 'sucursal', 'fecha_origen'], how='left')
    # NaN explícito donde no hubo ningún registro de inventario en la ventana
    matriz.loc[matriz['dias_con_registro_inv'].isna(), 'target_quiebre_binario'] = np.nan

    print(f"target de Nivel 2 = {diagnostico_inv['definicion_target']}")
    print(matriz['target_quiebre_binario'].value_counts(dropna=False))
    prev = matriz['target_quiebre_binario'].mean()
    print(f'prevalencia entre filas con registro: {prev:.2%}')
    assert 0 < prev < 1, 'el target de Nivel 2 sigue siendo de una sola clase -- volver al 06'
else:
    print('Nivel 2 NO supervisado (veredicto del 06): no se construye target_quiebre_binario.')
    print('El Nivel 2 se planteará como regla sobre el pronóstico cuantílico del Nivel 1.')

Nivel 2 NO supervisado (veredicto del 06): no se construye target_quiebre_binario.
El Nivel 2 se planteará como regla sobre el pronóstico cuantílico del Nivel 1.


## 11. Invariantes, exportación y huella

Los `assert` cubren precisamente las tres formas en que el intento anterior falló en
silencio: target nulo, ventanas solapadas y features estáticas variando dentro de un mismo
fold (que indicaría que el as-of se calculó mal).

In [13]:
assert matriz['target_demanda_15d'].notna().all(), 'hay nulos en el target de Nivel 1'
assert (matriz['target_demanda_15d'] >= 0).all(), 'target negativo: revisar devoluciones en el panel'
assert matriz['fecha_origen'].notna().all()

# Las ventanas no se solapan dentro de un fold (salvo los orígenes de referencia añadidos)
for fid, g in matriz.groupby('fold_id'):
    ori = sorted(g['fecha_origen'].unique())
    idxs = np.array([indice_habil[pd.Timestamp(o)] for o in ori])
    espaciados = np.diff(idxs)
    n_solapados = int((espaciados < HORIZONTE).sum())
    marcados = int(g.loc[g['es_origen_baseline'], 'fecha_origen'].nunique())
    assert n_solapados <= marcados, f'fold {fid}: hay {n_solapados} solapes no justificados'

# Una feature as-of debe ser constante por (par, fold): si varía, el motor as-of está mal
var_asof = (matriz.groupby(['codigo_item', 'sucursal', 'fold_id'])['frecuencia_as_of']
            .nunique().max())
assert var_asof == 1, 'frecuencia_as_of varía dentro de un fold: el cálculo as-of no es estático'

# Ninguna bandera con fuga se coló
assert not set(COND_CON_FUGA) & set(matriz.columns), 'una bandera con fuga temporal entró a la matriz'

print('Invariantes verificadas OK.')
print(f'\nmatriz: {matriz.shape}')
print(matriz.groupby(['fold_id', 'familia_modelo']).size().unstack(fill_value=0))

matriz.to_parquet(f'{DW}/matriz_as_of.parquet', index=False)
resumen_baseline.to_csv(f'{DW}/piso_interno_por_patron.csv')

huella = {
    'notebook': '07_matriz_as_of',
    'fecha_generacion': pd.Timestamp.now().isoformat(),
    'parametros': {'HORIZONTE': HORIZONTE, 'GAP_DIAS': GAP_DIAS,
                   'MIN_FRECUENCIA_ASOF': MIN_FRECUENCIA_ASOF,
                   'TOLERANCIA_EMPALME': TOLERANCIA_EMPALME,
                   'folds': [{k: (str(v.date()) if isinstance(v, pd.Timestamp) else v)
                              for k, v in f.items()} for f in FOLDS]},
    'entradas': registrar_huella(['panel_diario.parquet', 'calendario_habil.csv',
                                  'dim_producto_priorizado.parquet', 'efectos_calendario.csv',
                                  'calendario_eventos.csv', 'diagnostico_inventario.json',
                                  'piso_baseline_por_patron.csv']),
    'salidas': registrar_huella(['matriz_as_of.parquet', 'piso_interno_por_patron.csv']),
}
with open(f'{DW}/huella_07.json', 'w') as f:
    json.dump(huella, f, indent=2, ensure_ascii=False)
print('\nguardado: matriz_as_of.parquet, piso_interno_por_patron.csv, huella_07.json')

Invariantes verificadas OK.

matriz: (228823, 27)
familia_modelo  intermitente  suave
fold_id                            
1                      20245    560
2                      56485   1404
3                      25255    510
4                      53060   1020
5                      69216   1068



guardado: matriz_as_of.parquet, piso_interno_por_patron.csv, huella_07.json
